### Qualidade de Dados - Camada Prata

Nesta etapa avaliamos a tabela **cubesat_eps.prata.eps_telemetria** sob as dimensões de qualidade de dados exigidas pelo trabalho: **completude**, **consistência**, **unicidade**, e **acurácia**. O objetivo não é somente detectar problemas, mas sim documentar decisões: quando um 'problema' é na verdade um comportamento esperado no domínio (ex: um satélite em eclipse não gera energia solar - isso não é erro de dado, é física).

#### 1. Completude

A completude mede se os dados que deveriam existir realmente existem, ou se há lacunas (nulos/ausências) que podem enviesar a análise. Aqui isso é especialmente relevante porque estamos lidando com telemetria de sensores reais em órbita: falhas de sensor, perda de sincronismo do downlink ou o próprio evento de falha do painel solar podem se manifestar como valores nulos ou ausentes em colunas específicas.

A célula de código a seguir conta, para cada uma das 42 colunas da tabela, quantas linhas têm valor nulo. Isso nos dá um mapa inicial de onde os problemas de completude (se existirem) estão concentrados.

In [0]:
CATALOG = "cubesat_eps"

from pyspark.sql import functions as F

df_prata = spark.table(f"{CATALOG}.prata.eps_telemetria")

null_counts = df_prata.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_prata.columns])
null_counts.show(vertical=True, truncate=False)

-RECORD 0--------------
 time_stamp_s   | 0    
 tpy_raw        | 0    
 tpy_c          | 0    
 tpx_raw        | 0    
 tpx_c          | 0    
 tmz_raw        | 0    
 tmz_c          | 0    
 tmx_raw        | 0    
 tmx_c          | 0    
 tpz_raw        | 0    
 tpz_c          | 0    
 vpy_raw        | 0    
 vpy_mv         | 0    
 vpx_raw        | 0    
 vpx_mv         | 0    
 vmz_raw        | 0    
 vmz_mv         | 0    
 vmx_raw        | 0    
 vmx_mv         | 0    
 vpz_raw        | 0    
 vpz_mv         | 0    
 ipy_raw        | 0    
 ipy_ma         | 0    
 ipx_raw        | 0    
 ipx_ma         | 0    
 imz_raw        | 0    
 imz_ma         | 0    
 imx_raw        | 0    
 imx_ma         | 0    
 ipz_raw        | 0    
 ipz_ma         | 0    
 vbat_raw       | 0    
 vbat_v         | 0    
 ibat_raw       | 0    
 ibat_ma        | 0    
 tbat_raw       | 0    
 tbat_c         | 0    
 satellite_id   | 0    
 sheet_name     | 0    
 sheet_date     | 1112 
 is_ground_test 

Precisamos no entanto validar a causa dos nulos em "sheet_date". A função **parse_sheet_date()** retorna "None" para a data quando o nome da aba não bate com nenhum padrão de data conhecido - isso acontece de propósito nas abas de teste em solo do TSURU ("test1 w batt", test2 wo batt"), que não representam uma data de coleta em órbita. Nesses casos, **is_ground_test** é marcado como "True". Vamos confirmar que as duas contagens batem: se o número de nulos em "sheet_date" for igual ao número de linhas com "is_ground_test = true", isso comprova que os nulos são intencionais e não um erro de parsing.

In [0]:
df_prata.filter(F.col("sheet_date").isNull()).groupBy("is_ground_test").count().show()

+--------------+-----+
|is_ground_test|count|
+--------------+-----+
|          true| 1112|
+--------------+-----+



##### 2. Consistência

A consistência mede se os valores armazenados respeitam as regras esperadas de domínio: categorias válidas, faixas numéricas plausíveis, formatos corretos. Diferente de completude (que checa a ausência de dados), aqui o dado existe mas pode estar "errado" no sentido de não fazer sentido fisicamente ou logicamente. 

####### 2.1. Categorias válidas - satellite_id

Esperamos exatamente 4 valores possíveis para "satellite_id": NEPALISAT, RAAVANA, TSURU, e UGUISU (os nomes usados para nomear os arquivos de origem). Se aparecer qualquer valor fora desses 4 - por exemplo, por erro de digitação em alguma etapa do pipeline, isso indicaria um bug de consistência categória.

In [0]:
df_prata.groupBy("satellite_id").count().orderBy("satellite_id").show()

+------------+-----+
|satellite_id|count|
+------------+-----+
|   NEPALISAT| 3040|
|     RAAVANA| 2159|
|       TSURU|24378|
|      UGUISU| 3260|
+------------+-----+



###### 2.2. Faixa de datas válidas - sheet_date

A constelação BIRDS foi lançada e operou entre 2018 e os anos seguintes (TSURU segue ativo). Uma data de coleta fora dessa janela plausível (ex. antes do lançamento, ou no futuro) indicaria erro de parsing na função **parse_sheet_date()** por exemplo, uma aba cujo nome foi interpretado incorretamente como uma data válida quando não deveria ser

In [0]:
df_prata.filter(F.col("sheet_date").isNotNull()).agg(F.min("sheet_date").alias("data_minima"),F.max("sheet_date").alias("data_maxima")).show()

+-----------+-----------+
|data_minima|data_maxima|
+-----------+-----------+
| 2020-10-23| 2022-01-24|
+-----------+-----------+



###### 2.3. Faixa de valores - time_stamp_s

O "time_stamp_s" representa os segundos decorridos desde o início de uma sessão de coleta (uma aba = uma sessão). Não deveria existir valor negativo, e o valor máximo deve ser compatível com a duração razoável de uma sessão de coleta em órbita.

In [0]:
df_prata.agg(F.min("time_stamp_s").alias("min_time_stamp"),F.max("time_stamp_s").alias("max_time_stamp")).show()

+--------------+--------------+
|min_time_stamp|max_time_stamp|
+--------------+--------------+
|           0.0|       12362.2|
+--------------+--------------+



##### 3. Unicidade

A unicidade verifica se não há duplicação indevida de registros. Aqui, a chave natural de uma leitura de telemetria é a combinação (satélite, sessão de coleta, instante dentro da sessão) - ou seja, 'satellite_id' + 'sheet_name' + 'time_stamp_s'. Não deveria existir mais de uma linha com essa mesma combinação: isso indicaria duplicação de leitura (ex. mesma aba lida duas vezes durante a consolidação, ou erro de concatenação de DataFrames).

In [0]:
duplicatas = (df_prata.groupBy("satellite_id", "sheet_name", "time_stamp_s").count().filter(F.col("count") > 1))
print(f"Total de combinações duplicadas: {duplicatas.count()}")
duplicatas.show(10)

Total de combinações duplicadas: 0
+------------+----------+------------+-----+
|satellite_id|sheet_name|time_stamp_s|count|
+------------+----------+------------+-----+
+------------+----------+------------+-----+



##### 4. Acurácia / Outliers

A acurácia avalia se os valores calibrados fazem sentido dentro de limites físicos plausíveis (ex. temperatura de painel solar não deveria ser um valor exorbitante como 10.000 graus Celsisus, isso seria erro de calibração/parsing, não um dado real). Importante: nem todo valor extremo é erro. É possível que ocorram falhas de paineis, uma vez que um valor próximo de zero poderia ser justamente a falha que desejamos identificar, e não um problema de qualidade de dado. Por isso, nesta etapa, começamos olhando estatísticas descritivas (min/max/media/desvio padrão) das colunas calibradas, para termos uma visão geral antes de julgar o que é erro e o que é sinal real.

In [0]:
colunas_calibradas = [c for c in df_prata.columns if c.endswith(("_c", "_mv", "_ma", "_v"))]
df_prata.select(colunas_calibradas).describe().show(vertical=True, truncate=False)

-RECORD 0----------------------
 summary | count               
 tpy_c   | 32837               
 tpx_c   | 32837               
 tmz_c   | 32837               
 tmx_c   | 32837               
 tpz_c   | 32837               
 vpy_mv  | 32837               
 vpx_mv  | 32837               
 vmz_mv  | 32837               
 vmx_mv  | 32837               
 vpz_mv  | 32837               
 ipy_ma  | 32837               
 ipx_ma  | 32837               
 imz_ma  | 32837               
 imx_ma  | 32837               
 ipz_ma  | 32837               
 vbat_v  | 32837               
 ibat_ma | 32837               
 tbat_c  | 32837               
-RECORD 1----------------------
 summary | mean                
 tpy_c   | 2.0747345981667005  
 tpx_c   | 0.15702500228400754 
 tmz_c   | -1.6734445899442705 
 tmx_c   | -1.5899741145658841 
 tpz_c   | 1.4324143496665356  
 vpy_mv  | 3040.4243584980445  
 vpx_mv  | 3488.9702710357174  
 vmz_mv  | 3400.665205104001   
 vmx_mv  | 3365.8819222218663  
 vpz_mv 

###### 4.1. Investigando outlier em vbat_v

A tensão de bateria 'vbat_v' tem desvio padrão médio muito baixo (0,11V) em torno de uma média de 4,1V, mas o mínimo registrado é de 0V e o máximo é de 6,44V, que são valores muito distantes da distribuição típica. Vamos contar quantas linhas têm valores suspeitos e verificar quais satélites pertencem, para decidir se é erro de sensor/parsing ou evento real.

In [0]:
df_prata.filter((F.col("vbat_v") < 1.0) | (F.col("vbat_v") > 5.0)).groupBy("satellite_id", "is_ground_test").count().orderBy("satellite_id").show()

+------------+--------------+-----+
|satellite_id|is_ground_test|count|
+------------+--------------+-----+
|       TSURU|         false|    7|
|      UGUISU|         false|    1|
+------------+--------------+-----+



In [0]:
df_prata.filter((F.col("vbat_v") < 1.0) | (F.col("vbat_v") > 5.0)).select("satellite_id", "sheet_name", "sheet_date", "time_stamp_s", "vbat_v", "ibat_ma", "tbat_c").orderBy("satellite_id", "sheet_date").show(truncate=False)

+------------+-----------+----------+------------+------+-------+------+
|satellite_id|sheet_name |sheet_date|time_stamp_s|vbat_v|ibat_ma|tbat_c|
+------------+-----------+----------+------------+------+-------+------+
|TSURU       |March 31   |2021-03-31|4045.4      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |2021-08-26|5900.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |2021-08-26|5910.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |2021-08-26|5890.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |2021-08-26|5930.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |2021-08-26|5940.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |2021-08-26|5920.0      |0.0   |0.0    |0.0   |
|UGUISU      |13 Feb 2021|2021-02-13|5355.0      |6.44  |0.0    |0.0   |
+------------+-----------+----------+------------+------+-------+------+



In [0]:
df_prata.filter((F.col("ibat_ma") == 0) & (F.col("tbat_c") == 0)).count()

8

Olhando as 8 linhas em detalhe, um padrão fica claro: nas 7 linhas de TSURU os valores de bateria estão zerados simultaneamente. Fisicamente isso não faz sentido (tensão, corrente e temperatura da bateria não caem a zero ao mesmo tempo por acaso, e se a bateria realmente zerasse o satélite provavelmente perderia energia por completo). A explicação mais provável é a de que um frame de telemetria da bateria não chegou ou veio corrompido, e o valor 0 foi usado como preenchimento em vez de nulo. A linha de UGUISU segue o mesmo padrão, porém de forma invertida (tensão anormalmente alta, corrente e temperatura zeradas).

Essas linhas não serão removidas na camada prata para preservar a fidelidade aos dados brutos. O tratamento será feito na camada ouro, nulificando especificamente as colunas 'vbat_v', 'ibat_ma', e 'tbat_c' apenas nessas 8 linhas, preservando o resto da linha intacto.